<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB20_Case_Study_GMRT_Bathymetry_Interpolation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# NB20 - Class 20 -- Case Study: GMRT Bathymetry -- Predicting Seafloor Depth from Sparse Soundings

## Block 4: Proyectos -- Case Studies (continued)

`NB18` asked whether wind fields predict wave height. `NB19` asked whether a ship's route reveals its nationality. This class asks a third, structurally different real question: **if you only have a sparse set of real depth measurements, how well can you predict the seafloor depth in between them?**

This is not a hypothetical. `A full-coverage multibeam sonar survey of a coastal area is slow and expensive` -- vessels sail back and forth in overlapping lines, at a few knots, for days or weeks per square kilometer. Historically, and still today in many poorly-surveyed waters, nautical charts are built from **sparse line soundings**, and the depth in between those lines has to be estimated somehow. This class treats that estimation problem as a genuine spatial machine learning task, using real bathymetric data for the coastal waters around Cartagena -- the same coastline this course is taught on.


## Agenda (120 min)

| # | Section | Minutes |
|---|---|---|
| 1 | Why sparse-sounding interpolation matters for hydrographic surveying | 5 |
| 2 | Real data: the GMRT bathymetry service | 5 |
| 3 | Downloading and parsing a real bathymetry grid | 10 |
| 4 | Exploring the real data: a map of the Cartagena shelf | 20 |
| 5 | Framing the task: simulating a sparse survey | 5 |
| 6 | k-Nearest Neighbors regression (new technique) | 15 |
| 7 | Training and evaluating kNN vs. Random Forest vs. a naive baseline | 30 |
| 8 | The real experiment: accuracy vs. survey density | 20 |
| 9 | Honest interpretation and limitations | 5 |
| 10 | Class summary, homework, and what's next | 5 |

As in every class this course, these are approximate guides, not a script -- if a section runs short or long, that is fine.


---

## 1. Why sparse-sounding interpolation matters


`A nautical chart's usefulness depends entirely on how well it represents the real seafloor` -- a ship, and especially a submarine or a vessel with deep draft, needs to know where the water is too shallow long before it gets there. Producing that knowledge is expensive. A modern multibeam echosounder survey, run at a safe survey speed with the overlapping line coverage needed for full-bottom certainty, can take a survey vessel days to cover a few square kilometers. The **International Hydrographic Organization's S-44 standard** sets minimum sounding density requirements precisely because of this trade-off: denser surveys cost more, but sparser surveys risk missing a shoal or wreck between sounding lines.

Historically -- and still today in large areas of the world's coastal waters that have never been fully resurveyed with modern equipment -- charts are built from **sparse point or line soundings**, with the depth *between* them estimated by interpolation. This class treats that estimation problem the way this course has treated every other real problem: as something a model can be trained and honestly evaluated on, using real bathymetric data as the ground truth.

> **Further reading**: [Bathymetry (Wikipedia)](https://en.wikipedia.org/wiki/Bathymetry) | [Hydrographic survey (Wikipedia)](https://en.wikipedia.org/wiki/Hydrographic_survey) | [IHO S-44 standard, summary (Wikipedia)](https://en.wikipedia.org/wiki/International_Hydrographic_Organization)


---

## 2. Real data: the GMRT bathymetry service


**GMRT (Global Multi-Resolution Topography)** is a real, continuously-updated global bathymetry and topography synthesis maintained by the Marine Geoscience Data System at Lamont-Doherty Earth Observatory (Columbia University). It combines actual multibeam sonar survey data, wherever such surveys have been contributed to the archive, with satellite-altimetry-derived bathymetry estimates filling the gaps where no real survey exists yet.

**This is an important, honest caveat to keep in mind for the rest of this class**: in areas GMRT covers only with satellite-derived estimates (not real soundings), the "ground truth" this notebook trains and tests against `is itself a model's estimate, not a certified real measurement`. Coastal areas close to a major port -- like the one used in this class -- are more likely to have real survey contributions than the open ocean, but this cannot be verified from within this notebook alone.

GMRT exposes a free, no-registration **GridServer** web API that returns a real regional grid for any bounding box you request -- no API key, no `getpass`, no account, unlike every other Block 4 case study's data source so far. This class requests a real bounding box covering the Cartagena coastline and the adjacent Mediterranean shelf and slope, out to real depths of roughly 2,500 m.

> **Further reading**: [GMRT official site](https://www.gmrt.org/) | [Ryan et al. 2009, the GMRT synthesis (DOI)](https://doi.org/10.1029/2008GC002332) | [Multibeam echosounder (Wikipedia)](https://en.wikipedia.org/wiki/Multibeam_echosounder)


---

## 3. Downloading and parsing a real bathymetry grid


The cell below requests a real grid from the GMRT GridServer, in **Esri ASCII grid** format (`.asc`) -- a simple, fully self-describing text format: a 6-line header (grid size, corner coordinates, cell size, no-data value) followed by the elevation values themselves, one row of the grid per line of text, read top-to-bottom (north to south).

No login or key is needed, so this download either works directly or fails with a normal network error -- there is no credential step to get wrong.


In [ ]:
import urllib.request
import os

GMRT_URL = (
    "https://www.gmrt.org/services/GridServer"
    "?minlongitude=-1.6&maxlongitude=-0.6"
    "&minlatitude=37.3&maxlatitude=37.8"
    "&format=esriascii&resolution=low"
)

grid_path = "cartagena_bathymetry.asc"
if not os.path.exists(grid_path):
    urllib.request.urlretrieve(GMRT_URL, grid_path)

print(f"Downloaded {os.path.getsize(grid_path) / 1e6:.1f} MB to {grid_path}")


The next cell parses the header **without assuming its exact values** -- grid size and resolution can change slightly between requests as GMRT's synthesis is updated, so the code reads whatever `ncols`/`nrows`/`cellsize`/corner values the file actually reports, rather than hardcoding the numbers seen while building this notebook.


In [ ]:
import numpy as np

with open(grid_path) as f:
    header = {}
    for _ in range(6):
        key, val = f.readline().split()
        header[key.lower()] = val
    ncols = int(header["ncols"])
    nrows = int(header["nrows"])
    xllcorner = float(header["xllcorner"])
    yllcorner = float(header["yllcorner"])
    cellsize = float(header["cellsize"])
    nodata_value = float(header["nodata_value"])
    grid = np.loadtxt(f)

assert grid.shape == (nrows, ncols), f"Unexpected grid shape {grid.shape}, expected ({nrows}, {ncols})"
n_nodata = (grid == nodata_value).sum()

print(f"Grid shape: {grid.shape} (rows x cols)")
print(f"Cell size: {cellsize:.6f} degrees (~{cellsize * 111_000:.0f} m at this latitude)")
print(f"No-data cells: {n_nodata} / {grid.size}")
print(f"Elevation range: {grid.min():.1f} m to {grid.max():.1f} m (negative = below sea level)")


Esri ASCII grids store rows **north to south** (the first row in the file is the northernmost). The cell below reconstructs the real latitude and longitude of every cell from the header alone, then flattens the grid into a tidy DataFrame -- the same "flatten a spatial grid into rows" pattern `NB18` used for its ERA5 wind/wave grids, applied here to a bathymetry grid instead of a weather grid.


In [ ]:
import pandas as pd

lons = xllcorner + cellsize * np.arange(ncols)
lats = yllcorner + cellsize * np.arange(nrows)[::-1]  # first row = northernmost

lon_grid, lat_grid = np.meshgrid(lons, lats)

bathy = pd.DataFrame({
    "longitude": lon_grid.ravel(),
    "latitude": lat_grid.ravel(),
    "elevation_m": grid.ravel(),
})
bathy = bathy[bathy["elevation_m"] != nodata_value].reset_index(drop=True)

print(f"{len(bathy):,} real grid cells with valid elevation data")
bathy.head()


---

## 4. Exploring the real data: a map of the Cartagena shelf


`elevation_m` is positive on land and negative underwater -- `this single real dataset covers both the coastline and the seafloor`. The next cell splits it into `land` (elevation > 0) and `sea` (elevation <= 0, the actual bathymetry this class cares about) and reports the real proportions and depth range, before plotting anything.


In [ ]:
sea = bathy[bathy["elevation_m"] <= 0].copy()
sea["depth_m"] = -sea["elevation_m"]
land = bathy[bathy["elevation_m"] > 0]

print(f"Sea cells: {len(sea):,} ({len(sea) / len(bathy):.1%})")
print(f"Land cells: {len(land):,} ({len(land) / len(bathy):.1%})")
print(f"Depth range: {sea['depth_m'].min():.1f} m to {sea['depth_m'].max():.1f} m")
print(sea["depth_m"].describe())


Following the same rule this course has used since `NB18`'s spatial residual map -- any real lat/lon data gets plotted on an actual geographic map with `cartopy`, not a bare scatter plot with unlabeled axes.


In [ ]:
%pip install -q cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(9, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
sc = ax.scatter(
    bathy["longitude"], bathy["latitude"], c=bathy["elevation_m"],
    cmap="terrain", s=2, transform=ccrs.PlateCarree(), vmin=-2500, vmax=900,
)
ax.coastlines(resolution="10m")
ax.add_feature(cfeature.BORDERS, linestyle=":")
gl = ax.gridlines(draw_labels=True, linestyle="--", alpha=0.4)
gl.top_labels = False
gl.right_labels = False
plt.colorbar(sc, label="Elevation (m) -- negative = depth below sea level", shrink=0.8)
plt.title("Real GMRT elevation/bathymetry -- Cartagena coast and Mediterranean shelf")
plt.show()


A histogram makes the real depth distribution concrete before mapping it spatially:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(sea["depth_m"], bins=40, color="steelblue", edgecolor="white")
ax.set_xlabel("Depth (m)")
ax.set_ylabel("Number of grid cells")
ax.set_title("Real depth distribution -- Cartagena shelf and slope")
plt.show()


---

## 5. Framing the task: simulating a sparse survey


The GMRT grid above is treated from here on as **ground truth** -- every sea cell has a real (or GMRT-estimated, per the Section 2 caveat) depth value. The genuine question is: if a survey vessel had only sampled a small, sparse fraction of these points -- as a real historical or budget-limited survey would -- how accurately could the *rest* of the depths be predicted?

The cell below simulates this by taking a small random sample of sea cells as the "known soundings" (the training set) and holding out the remaining, much larger set of sea cells as genuinely unseen points whose real depth the model never gets to train on -- evaluated only afterward, the same discipline as every genuine holdout in this course since `NB13`.


In [ ]:
from sklearn.model_selection import train_test_split

SURVEY_FRACTION = 0.02  # 2% of sea cells "surveyed" -- a deliberately sparse starting point

known, unknown = train_test_split(sea, train_size=SURVEY_FRACTION, random_state=42)

X_known, y_known = known[["longitude", "latitude"]], known["depth_m"]
X_unknown, y_unknown = unknown[["longitude", "latitude"]], unknown["depth_m"]

print(f"Simulated survey: {len(known):,} known soundings ({SURVEY_FRACTION:.1%} of all sea cells)")
print(f"Held out to test against: {len(unknown):,} real depth values the model never trains on")


---

## 6. k-Nearest Neighbors regression (new technique)


Every model this course has used so far -- linear/logistic regression, decision trees and their ensembles, SVM, neural networks -- learns an explicit function from the training data and then applies that function to new points. **k-Nearest Neighbors (kNN) regression** works completely differently: it does not "learn" anything in that sense. To predict a new point's value, it simply finds the `k` closest training points (by real geographic distance, here) and averages their known values.

This is an unusually good conceptual fit for spatial interpolation specifically: two points close together on the seafloor usually *do* have similar depths (the seafloor is mostly a smooth surface, only rarely disrupted by cliffs, wrecks, or reefs) -- `exactly the assumption kNN makes explicit and exactly what a chart-maker interpolating between sounding lines is implicitly relying on too`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(7)
neighbors_x = rng.uniform(0, 10, 25)
neighbors_y = rng.uniform(0, 10, 25)
query_x, query_y = 5.0, 5.0

dist = np.hypot(neighbors_x - query_x, neighbors_y - query_y)
k = 5
nearest = np.argsort(dist)[:k]

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(neighbors_x, neighbors_y, c="steelblue", label="known soundings")
ax.scatter(neighbors_x[nearest], neighbors_y[nearest], c="orange", s=90,
           edgecolor="black", label=f"{k} nearest neighbors", zorder=3)
ax.scatter([query_x], [query_y], c="red", marker="*", s=250,
           edgecolor="black", label="query point (unknown depth)", zorder=4)
radius = dist[nearest].max()
circle = plt.Circle((query_x, query_y), radius, fill=False, linestyle="--", color="orange")
ax.add_patch(circle)
ax.set_aspect("equal")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
ax.set_title(f"k-Nearest Neighbors (k={k}): predict from the {k} closest known points")
plt.tight_layout()
plt.show()


For regression, kNN's prediction is the **average** (optionally distance-weighted, so closer points count more) of the `k` neighbors' known values. `k` itself is a hyperparameter with a real trade-off, just like tree depth in `NB08`: a small `k` follows local detail closely but is sensitive to noise in individual soundings; a large `k` smooths over real local features (a rock outcrop, a trench) that a chart-maker actually needs to see.

> **Further reading**: [k-nearest neighbors algorithm (Wikipedia)](https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm) | [scikit-learn `KNeighborsRegressor` docs](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html) | [Spatial interpolation / kriging (Wikipedia)](https://en.wikipedia.org/wiki/Kriging)


---

## 7. Training and evaluating kNN vs. Random Forest vs. a naive baseline


Three predictors are compared on the same sparse survey from Section 5, all trained only on `X_known`/`y_known` and evaluated only on the held-out `X_unknown`/`y_unknown`:

- A **naive baseline**: predict every unknown point's depth as its single nearest known sounding's exact depth (`k=1`, unweighted) -- the simplest thing a chart-maker could do by hand, and the bar any real model needs to clear.
- **kNN regression** (`k=8`, distance-weighted) -- this class's new technique.
- **Random Forest regression** -- already familiar from `NB07`/`NB10`/`NB18`, included so the new spatial-specific technique is judged against a strong general-purpose model, not only against the naive baseline.


In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

models = {
    "Naive nearest-sounding": KNeighborsRegressor(n_neighbors=1),
    "kNN (k=8, distance-weighted)": KNeighborsRegressor(n_neighbors=8, weights="distance"),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
}

results = {}
predictions = {}
for name, model in models.items():
    model.fit(X_known, y_known)
    pred = model.predict(X_unknown)
    predictions[name] = pred
    mae = mean_absolute_error(y_unknown, pred)
    rmse = mean_squared_error(y_unknown, pred) ** 0.5  # sqrt(MSE): `squared=False` was removed in newer scikit-learn
    results[name] = {"MAE (m)": mae, "RMSE (m)": rmse}

results_df = pd.DataFrame(results).T
print(f"Evaluated on {len(y_unknown):,} real held-out depth soundings (survey fraction {SURVEY_FRACTION:.1%})")
results_df


A bar chart makes the three-way comparison easier to read than the table alone:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, metric in zip(axes, ["MAE (m)", "RMSE (m)"]):
    ax.bar(results_df.index, results_df[metric], color=["steelblue", "darkorange", "seagreen"])
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


A single error number hides *where* a model is wrong. The map below plots the real prediction error (predicted minus true depth) for the best-performing model at every held-out point -- a systematic pattern (errors clustering near the coast, or near the deepest points) would say something real about where sparse-survey interpolation is trustworthy and where it is not; scattered, unstructured error would suggest the model is doing about as well as the sparse data allows.


In [ ]:
best_name = results_df["RMSE (m)"].idxmin()
error = predictions[best_name] - y_unknown.values
print(f"Best model on this run: {best_name}")

fig = plt.figure(figsize=(9, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
sc = ax.scatter(
    X_unknown["longitude"], X_unknown["latitude"], c=error,
    cmap="coolwarm", s=3, vmin=-100, vmax=100, transform=ccrs.PlateCarree(),
)
ax.scatter(X_known["longitude"], X_known["latitude"], c="black", s=4,
           marker="x", transform=ccrs.PlateCarree(), label="known soundings")
ax.coastlines(resolution="10m")
gl = ax.gridlines(draw_labels=True, linestyle="--", alpha=0.4)
gl.top_labels = False
gl.right_labels = False
plt.colorbar(sc, label="Prediction error (m): predicted - true depth", shrink=0.8)
ax.legend(loc="lower left")
plt.title(f"{best_name}: real prediction error at each held-out point")
plt.show()


**Try it yourself**: a predicted-vs-actual scatter for the best model is a different, complementary view from the spatial error map above — do the points hug the diagonal closely, or is there a systematic bias (e.g., consistently under-predicting depth)?

In [ ]:
best_pred = predictions[best_name]

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_unknown, best_pred, alpha=0.3, s=5)
lims = [y_unknown.min(), y_unknown.max()]
ax.plot(lims, lims, "r--", label="Perfect prediction")
ax.set_xlabel("Actual depth (m)")
ax.set_ylabel("Predicted depth (m)")
ax.set_title(f"{best_name}: predicted vs. actual depth")
ax.legend()
plt.show()


---

## 8. The real experiment: accuracy vs. survey density


Section 5 fixed the survey density at a single, arbitrarily-chosen 2%. The genuinely useful question for planning a real survey is different: **how much does accuracy actually improve as more of the seafloor gets surveyed?** The cell below repeats the exact same train/evaluate procedure from Section 7 -- kNN only, to keep this loop fast -- at several real survey-fraction levels, and plots RMSE against survey density.


In [ ]:
fractions = [0.002, 0.005, 0.01, 0.02, 0.05, 0.10]
density_results = []

for frac in fractions:
    known_f, unknown_f = train_test_split(sea, train_size=frac, random_state=42)
    knn = KNeighborsRegressor(n_neighbors=8, weights="distance")
    knn.fit(known_f[["longitude", "latitude"]], known_f["depth_m"])
    pred_f = knn.predict(unknown_f[["longitude", "latitude"]])
    rmse_f = mean_squared_error(unknown_f["depth_m"], pred_f) ** 0.5  # sqrt(MSE): `squared=False` was removed in newer scikit-learn
    density_results.append({"survey_fraction": frac, "n_known": len(known_f), "RMSE_m": rmse_f})
    print(f"Survey fraction {frac:>6.1%}  ({len(known_f):>5,} soundings)  ->  RMSE = {rmse_f:6.2f} m")

density_df = pd.DataFrame(density_results)


The printed numbers above already show the trend; the plot below makes the shape of the trade-off (does accuracy improve steadily, or is there a point of diminishing returns?) easier to read at a glance.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(density_df["survey_fraction"] * 100, density_df["RMSE_m"], marker="o")
ax.set_xlabel("Survey coverage (% of sea cells actually sounded)")
ax.set_ylabel("kNN RMSE on held-out depth (m)")
ax.set_title("Real trade-off: sounding density vs. interpolation accuracy")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Try it yourself**: quantify what the plot shows — the real RMSE improvement gained per extra 1% of survey coverage, at each step. Where does the payoff from surveying more clearly start shrinking?

In [ ]:
density_df["RMSE_improvement_m"] = -density_df["RMSE_m"].diff()
density_df["coverage_increase_pct"] = density_df["survey_fraction"].diff() * 100
density_df["improvement_per_1pct_coverage"] = density_df["RMSE_improvement_m"] / density_df["coverage_increase_pct"]
density_df.round(3)


---

## 9. Honest interpretation and limitations


A few real limitations of this exercise are worth stating plainly, not hidden:

- **The random-sparsity simulation is not how real surveys are planned.** A real hydrographic survey follows planned transect lines, not a uniformly-random scatter of points -- random sampling is the easiest thing to simulate here, but it is a genuinely more favorable case than a real line survey, which leaves long, systematic gaps between lines rather than small gaps everywhere.
- **The "ground truth" itself has the caveat raised in Section 2.** Where GMRT's own coverage in this area comes from real multibeam data, this evaluation is against real seafloor depth. Where it comes from GMRT's own satellite-derived estimate, this notebook is really testing "how well does kNN reproduce another model's estimate" -- a different, weaker claim. This cannot be resolved without an independent, certified source for this exact area.
- **Sparse-point interpolation, however good the RMSE looks, cannot recover features smaller than the survey's own spacing** -- a shipwreck, an isolated rock pinnacle, or a narrow channel will simply not appear if no sounding ever landed on it. A low average error does not mean an interpolated chart is safe to navigate by; real hydrographic practice requires either dense-enough coverage or a documented uncertainty/hazard margin, exactly the kind of number this notebook's RMSE curve could inform but not replace.

None of this makes the survey-density experiment above meaningless -- it is a real, honestly-evaluated relationship between sounding density and achievable accuracy on real terrain. It is simply not a substitute for an actual hydrographic survey standard.


---

## Class summary

- Real bathymetry for the Cartagena coast, downloaded live and with no login from GMRT's public GridServer -- the first Block 4 data source this course needed no credential for at all.
- Framed a genuine naval-engineering cost/accuracy trade-off (hydrographic survey density) as a real, honestly-evaluated spatial machine learning task.
- **k-Nearest Neighbors regression**, a new technique for this course, introduced specifically because its "nearby points are similar" assumption matches spatial interpolation naturally.
- Compared kNN against a naive baseline and against `NB07`/`NB10`/`NB18`'s already-familiar Random Forest, on a genuine held-out set of real depth values the model never trained on.
- The real survey-density experiment (Section 8) is the kind of concrete number an actual survey-planning decision could use -- and its own honest limitations (Section 9) are as much a part of the result as the RMSE curve itself.

## For the next class

Another Block 4 case study -- topic to be confirmed, continuing the same pattern: a real dataset, a real question, and a genuine holdout wherever the question calls for one.

## Homework / Practice Ideas

1. Repeat Section 8's density experiment with Random Forest instead of kNN -- does the same accuracy-vs-density relationship hold, or does one method need less data than the other to reach a given RMSE?
2. Change `n_neighbors` in Section 7's kNN model (try 1, 3, 8, 30) at a fixed survey fraction -- plot RMSE against `k` and connect the result back to Section 6's local-detail-vs-smoothing trade-off.
3. Instead of a uniformly-random "survey," simulate a real line-survey pattern (e.g. keep only points within a fixed distance of a few straight north-south lines) at the same overall point count as one of Section 8's fractions -- does interpolation accuracy get worse, matching the Section 9 caveat about line surveys being harder than random sampling?
4. Request a GMRT grid for a different real coastal area (change the bounding box in Section 3) and re-run the full notebook -- does the accuracy-vs-density curve look similar, or does local seafloor terrain (a steep shelf edge vs. a gentle slope) change the relationship?
5. Add `elevation_m > 0` (land) points back into a combined interpolation task -- does mixing land and sea points help or hurt predicting depth near the coastline, where the two regimes meet?

> ***As always: a real dataset with an honestly-stated limitation teaches more than a clean one that hides its assumptions.***
